## Import packages

In [ ]:
import astropy
from pylab import *
from astropy.io import fits
from astropy.time import Time
from PyAstronomy import pyasl
from astropy import units as u
from astropy.coordinates import SkyCoord, EarthLocation
from astropy.table import Table, QTable 

## Read in FITS files

In [ ]:
hdul = fits.open('../../../data/he1158-2313_uncor.fits')
head = hdul[1].header
data = hdul[1].data
wvl2, flx2 = data['wavelength'],data['norm_flux']      # option 2 for this file

wvl1, flx1 = pyasl.read1dFitsSpec("../../../data/HD140283.fits")

# merged orders are not strictly monotonic
_o = np.argsort(wvl2, kind='stable'); wvl2, flx2 = np.asarray(wvl2,float)[_o], np.asarray(flx2,float)[_o]
data  # check on what the arrays are called in the star spectrum


## Plotting observed spectrum and reference spectrum near the H$_{\alpha}$ line

In [ ]:
fig  = plt.figure(figsize=(12,7))
#plt.plot(wvl1, flx1, 'b--',label="HD 122563")  #reference star
plt.plot(wvl1, flx1, 'b--',label="HD 140283")  #reference star

#plt.plot(wvl2, flx2, 'b.-',label="HE 1158-2313")   #target
plt.plot(wvl2, flx2, 'r.-',label="HE 1158-2313")   #target

# lines below are used to control the tick markers
plt.tick_params(axis="both",bottom=True, top=True, left=True, right=True, length=7,direction="in")
plt.tick_params(which='minor',direction="in",bottom=True, top=True, left=True, right=True, length=4, color='r')
plt.minorticks_on()
plt.tick_params(axis="both",which='major', direction="in", labelsize=15)
# lines below are used to add labels
plt.ylabel('Flux', fontsize=14)
plt.xlabel('$\lambda$ ($\AA$)', fontsize=14)
# lines below are used to control the x- and y-axis limits. This wavelength range shows the Mg triplet lines.
xlim([6555,6575])
ylim([0.0,1.1])
# line below is used to show the legend
plt.legend(loc="lower right", fontsize=16)
# line below is used to show the figure
plt.show()


## Apply the Doppler shift to the spectum, i.e. shift the spectrum to rest

In [ ]:
# The velocity comes from part4-2-2_calculate_geocentric_rv.ipynb rather than being typed in,
# so the two notebooks cannot disagree. Her formula below adds RV, so RV must be the negative
# of the measured (positive) velocity to bring the spectrum back to rest.
import json
# §4.2.2 was run against both templates the worksheet offers. HD 122563 is adopted: it is a
# cool metal-poor giant, the closest physical match to HE 1158-2313 (HD 140283 is a subgiant).
# The two agree to 0.10 km/s, so the choice barely matters - see the summary notebook.
ADOPTED_TEMPLATE = "HD122563"
V_GEO = json.load(open(f"results/part4-2-2_geocentric_rv_{ADOPTED_TEMPLATE}.json"))["v_geo_kms"]
RV = -V_GEO
print(f"measured v_geo = {V_GEO:+.2f} km/s  ->  applying RV = {RV:+.2f} km/s")

In [ ]:
#flux_new, wvl_new = pyasl.dopplerShift(wvl2, flx2, RV , edgeHandling="firstlast")

In [ ]:
# Her line, kept for reference: this is the FIRST-ORDER Doppler shift.
wvl_new_firstorder = wvl2 + RV/299792.458*wvl2
# At ~350 km/s the first-order form differs from the exact one by 0.0070 A at 5183.6 A
# = 0.41 km/s, which is larger than the measurement precision - so the exact form is what
# gets saved. The difference is recorded in the results file.
wvl_new = wvl2 / (1 + V_GEO/299792.458)
print(f"first-order vs exact at 5183.62 A: "
      f"{(wvl_new_firstorder - wvl_new)[np.argmin(abs(wvl2 - 5183.62*(1+V_GEO/299792.458)))]:+.4f} A")

In [ ]:
# Plot the shifted observed spectrum again and the reference star:
fig  = plt.figure(figsize=(12,7))

#plt.plot(wvl1   , flx1    , 'b--',label="HD 122563")
plt.plot(wvl1   , flx1    , 'b--',label="HD 140283")

#plt.plot(wvl_new, flux_new, 'rp-',label="HE 1158-2313")
#plt.plot(wvl_new, flux_new, 'rp-',label="HE 1158-2313")

plt.plot(wvl_new, flx2, 'gp-',label="HE 1158-2313")


plt.tick_params(axis="both",bottom=True, top=True, left=True, right=True, length=7,direction="in")
plt.tick_params(which='minor',direction="in",bottom=True, top=True, left=True, right=True, length=4, color='r')
plt.minorticks_on()
plt.tick_params(axis="both",which='major', direction="in", labelsize=15)
plt.ylabel('Flux', fontsize=14)
plt.xlabel('$\lambda$ ($\AA$)', fontsize=14)
xlim([6555,6570])
ylim([0.0,1.1])
plt.legend(loc="lower right", fontsize=16)
plt.show()

# TASK (optional): Can you plot the observed spectrum and the overplot the shifted 
# version? (with or without the reference spectrum)

##  Saving the rest frame (=shifted) spectrum to a new file for later

In [ ]:
a,b = wvl_new, flx2
X   = np.column_stack((a,b))
X   = X[~np.isnan(X).any(axis=1), :]
np.savetxt("../../../data/he1158-2313_rest.txt", X, delimiter="   ", header='',comments='')

# one-word deviation from her line: name it norm_flux, matching the input file and what
# Worksheet 3 will expect, rather than 'flux'
t = Table([a, b], names=('wavelength', 'norm_flux'))
t.write("../../../data/he1158-2313_rest.fits", overwrite=True)



## Finally, calculate the heliocentric radial velocity correction to get the final rad. velocity

## How to enter the header information:

In [ ]:
# from part4-4_header_information.ipynb, so the header is parsed in one place only
H = json.load(open("results/part4-4_header.json"))
RA              = H["RA"]
DEC             = H["DEC"]
telescope       = 'Las Campanas Observatory'
obs_date        = H["UT-DATE"]
ut_start        = H["UT-START"]
print(RA, DEC, obs_date, ut_start)

In [ ]:
c        = SkyCoord(ra=RA,dec=DEC, unit=(u.hourangle, u.deg))
ra2000   = c.ra.deg
dec2000  = c.dec.deg


In [ ]:
print("RA in hours = " , RA , ";  RA in Degrees =" , ra2000)
print("DEC in angle = ", DEC, ";  DEC in Degrees =", dec2000)


In [ ]:
location = EarthLocation.of_site(telescope)
sc = SkyCoord(ra=ra2000*u.deg, dec=dec2000*u.deg)


In [ ]:
heliocorr = sc.radial_velocity_correction('heliocentric', obstime=Time(obs_date), location=location)
heliocorr.to(u.km/u.s).value  # express correction in km/s
heliocorr=round(heliocorr.to(u.km/u.s).value, 2) 

print(heliocorr, 'km/s')


In [ ]:
print("Heliocentric correction of ", 'HE 1158-2313', "=", heliocorr, "km/s")


## Calculate the final line-of-sight heliocentric velocity (in km/s):

In [ ]:
RV_value = V_GEO                      # the measured geocentric velocity, not a typed-in number

Line_of_sight_velocity =  RV_value + heliocorr


In [ ]:
print("Final heliocentric velocity of",'J0449-5656', "=", round(Line_of_sight_velocity, 1), " km/s ")


In [ ]:
#SOME EXTRA PLOTS - JUST FOR FUN :)

In [ ]:
# Plot the shifted observed spectrum again as well as the reference star:

fig  = plt.figure(figsize=(12,7))
plt.plot(wvl_new, flx2, 'gp-',label="HE 1158-2313") # this is the restframe spectrum of your star now!  
plt.plot(wvl1   , flx1, 'b--',label="HD 140283") # comparison/reference star

plt.tick_params(axis="both",bottom=True, top=True, left=True, right=True, length=7,direction="in")
plt.tick_params(which='minor',direction="in",bottom=True, top=True, left=True, right=True, length=4, color='r')
plt.minorticks_on()
plt.tick_params(axis="both",which='major', direction="in", labelsize=15)
plt.ylabel('Flux', fontsize=14)
plt.xlabel('$\lambda$ ($\AA$)', fontsize=14)
xlim([5170,5190])
#xlim([4095,4110])
#xlim([4222,4230])
#xlim([4205,4227])
#xlim([4530,4559])
#xlim([4127,4132])
ylim([0.0,1.1])
plt.legend(loc="lower right", fontsize=16)
plt.show()

# TASK (optional): Can you plot the observed spectrum and then overplot the shifted 
# version? (with or without the reference spectrum)

# Thinking question: Why do these two spectra look so different? Which line are you looking at? 


### The observation time, and saving results

In [ ]:
from astropy.coordinates import EarthLocation
site = EarthLocation(lat=float(H["SITELAT"])*u.deg, lon=float(H["SITELONG"])*u.deg,
                     height=float(H["SITEALT"])*u.m)      # from the header, not the registry
sc2 = SkyCoord(RA, DEC, unit=(u.hourangle, u.deg))

def corr(t):
    return sc2.radial_velocity_correction('heliocentric', obstime=Time(t),
                                          location=site).to(u.km/u.s).value

helio_date  = corr(obs_date)                                  # AF's call: 00:00 UT
helio_start = corr(f"{obs_date}T{ut_start}")                   # what 4.4 actually asks for
helio_mid   = corr(Time(f"{obs_date}T{ut_start}") + float(H["EXPTIME"])/2*u.s)
print(f"00:00 UT (AF's call)  : {helio_date:+.3f} km/s")
print(f"UT-START {ut_start}     : {helio_start:+.3f} km/s")
print(f"mid-exposure          : {helio_mid:+.3f} km/s")
print(f"difference (date vs UT-START): {helio_date - helio_start:+.3f} km/s")

out = {
  "v_geo_kms": V_GEO,
  "rest_files": ["data/he1158-2313_rest.fits", "data/he1158-2313_rest.txt"],
  "shift_formula": "exact lambda/(1+v/c); AF's first-order lambda(1+RV/c) differs by 0.41 km/s at 350 km/s",
  "helio_corr_kms": {"at_00UT_AF_call": round(float(helio_date), 3),
                     "at_UT_START": round(float(helio_start), 3),
                     "at_mid_exposure": round(float(helio_mid), 3)},
  "helio_corr_adopted_kms": round(float(helio_start), 3),
  "v_helio_if_added_kms": round(float(V_GEO + helio_start), 2),
  "_note": "AF's apply_radial_velocity_af.ipynb, adapted: RV read from 4.2.2, header from 4.4, exact shift"}
json.dump(out, open("results/part4-3_shift_and_helio.json", "w"), indent=2)
print(); print(json.dumps(out, indent=2))